Aqui será feito o tratamento do banco de dados escolhido para a realização do projeto de algoritmos de busca, mais especificamente, pro nosso grupo, será o algoritmo genético.

O problema escolhido pode ser descrito como: saindo de natal, quero ir em uma viagem de avião e passar por todos os estados do país, e depois retornar pra natal, com o critério de avaliação sendo a distancia percorrida, tentando encontrar o melhor caminho.

In [ ]:
# importando as bibliotecas, lendo o dataset bruto e mostrando ele:
import pandas as pd
import numpy as np

aeroportos_global = pd.read_csv("airports.csv")
display(aeroportos_global)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NaN,US,US-AL,Harvest,no,NaN,NaN,00AL,00AL,NaN,NaN,NaN
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80.0,NaN,US,US-AK,King Salmon,no,NaN,NaN,00AN,00AN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84693,32753,ZYYY,medium_airport,Shenyang Dongta Airport,41.784354,123.496308,157.0,AS,CN,CN-21,"Dadong, Shenyang",no,ZYYY,NaN,ZYYY,NaN,NaN,NaN,"东塔机场, SHE"
84694,46378,ZZ-0001,heliport,Sealand Helipad,51.894444,1.482500,40.0,EU,GB,GB-ENG,Sealand,no,NaN,NaN,NaN,NaN,http://www.sealandgov.org/,https://en.wikipedia.org/wiki/Principality_of_...,Roughs Tower Helipad
84695,307326,ZZ-0002,small_airport,Glorioso Islands Airstrip,-11.584278,47.296389,11.0,AF,TF,TF-U-A,Grande Glorieuse,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84696,346788,ZZ-0003,small_airport,Fainting Goat Airport,32.110587,-97.356312,690.0,NaN,US,US-TX,Blum,no,NaN,NaN,87TX,87TX,NaN,NaN,NaN


In [4]:
#tratando os vazios pro pandas:
aeroportos_global_trat = aeroportos_global.replace([None, "NULL", "null", "NA", "N/A", np.nan], pd.NA)

In [ ]:
# verificando os tipos:
print(aeroportos_global_trat.dtypes)

# tudo parece adequado

id                     int64
ident                    str
type                     str
name                     str
latitude_deg         float64
longitude_deg        float64
elevation_ft          object
continent                str
iso_country              str
iso_region               str
municipality             str
scheduled_service        str
icao_code                str
iata_code                str
gps_code                 str
local_code               str
home_link                str
wikipedia_link           str
keywords                 str
dtype: object


In [6]:
# verificando se há duplicatas exatas:
print("Duplicatas exatas:", aeroportos_global_trat.duplicated().sum())

Duplicatas exatas: 0


In [9]:
# verificando se há duplicatas nos identificadores principais:
print(f"Duplicatas em id:", aeroportos_global_trat['id'].duplicated().sum())
print(f"Duplicatas em ident:", aeroportos_global_trat['ident'].duplicated().sum())
print(f"Duplicatas em name:", aeroportos_global_trat['name'].duplicated().sum())

Duplicatas em id: 0
Duplicatas em ident: 0
Duplicatas em name: 4558


In [10]:
# função pra me mostrar as duplicatas:
def mostrar_duplicatas(df, chave_busca=None):
    """
    Exibe as linhas duplicadas de um DataFrame.
    Se chave_busca for None, procura por linhas 100% idênticas.
    Se chave_busca for uma lista de colunas, procura por repetições nessas chaves.
    """
    if chave_busca is None:
        # Busca duplicatas exatas (todas as colunas iguais)
        duplicatas = df[df.duplicated(keep=False)]
        tipo = "Exatas (linha inteira)"
        coluna_ordenacao = df.columns[0] # Ordena pela primeira coluna por padrão
    else:
        # Busca duplicatas em colunas específicas (ex: IDs)
        duplicatas = df[df.duplicated(subset=chave_busca, keep=False)]
        tipo = f"Nas colunas: {chave_busca}"
        coluna_ordenacao = chave_busca
    
    qtd_linhas = len(duplicatas)
    
    if qtd_linhas == 0:
        print(f"✅ Nenhuma duplicata encontrada! | Tipo de busca: {tipo}")
    else:
        print(f"⚠️ ATENÇÃO: {qtd_linhas} linhas envolvidas em duplicações! | Tipo de busca: {tipo}")
        print("-" * 60)
        # O display exibe a tabela formatada e o sort_values coloca as cópias lado a lado
        display(duplicatas.sort_values(by=coluna_ordenacao))

# como foram encontradas duplicatas em 'name', eu quero observa-las pra ver se é necessária correção       
mostrar_duplicatas(aeroportos_global_trat, chave_busca='name')

⚠️ ATENÇÃO: 7164 linhas envolvidas em duplicações! | Tipo de busca: Nas colunas: name
------------------------------------------------------------


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
12755,43214,BGAG,heliport,Aappilattoq Heliport,72.887030,-55.596287,42.0,NaN,GL,GL-AV,Aappilattoq,yes,NaN,AOQ,BGAG,AAP,NaN,https://en.wikipedia.org/wiki/Aappilattoq_Heli...,NaN
12758,44046,BGAQ,heliport,Aappilattoq Heliport,60.148357,-44.286916,30.0,NaN,GL,GL-KU,Aappilattoq,yes,NaN,QUV,BGAQ,NaN,NaN,https://en.wikipedia.org/wiki/Aappilattoq_Heli...,Augpilagtoq
28153,31400,FZJF,small_airport,Aba Airport,3.859556,30.253676,3051.0,AF,CD,CD-HU,Aba,no,FZJF,NaN,FZJF,NaN,NaN,https://en.wikipedia.org/wiki/Aba_Airport,NaN
61583,37059,SNDH,small_airport,Aba Airport,-12.162858,-45.033695,1682.0,SA,BR,BR-BA,Barreiras,no,NaN,NaN,SNDH,NaN,NaN,NaN,NaN
67921,14610,US-10808,closed,Abbott Airport,42.322943,-99.755845,2560.0,NaN,US,US-NE,Long Pine,no,NaN,NaN,NaN,NaN,NaN,NaN,83NE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67339,11841,US-10278,closed,Zimmerman Airport,45.793598,-96.300102,1045.0,NaN,US,US-MN,Herman,no,NaN,NaN,NaN,NaN,NaN,NaN,50MN
11623,507989,AU-0581,closed,dont know if this is an airport,-22.368055,143.026875,<NA>,OC,AU,AU-QLD,Winston,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11622,507988,AU-0580,closed,dont know if this is an airport,-22.368055,143.026875,<NA>,OC,AU,AU-QLD,Winston,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20363,1554,CSR5,heliport,Île d'Orléans Heliport,46.990799,-70.826698,170.0,NaN,CA,CA-QC,Île d'Orléans,no,NaN,NaN,CSR5,CSR5,NaN,NaN,SR5


Talvez não seja relevante pra mim, se não envolver os principais aeroportos do brasil, então vou deixar pra verificar novamente depois de tirar os dados do resto do mundo.

In [12]:
# eliminando todos os aeroportos que não sejam do brasil:
aeroportos_brasil = aeroportos_global_trat[aeroportos_global_trat['iso_country'] == 'BR']
display(aeroportos_brasil)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
12978,41842,BR-0001,small_airport,Tatamborá Flying Field,-23.888048,-45.446824,500.0,SA,BR,BR-SP,"Ponta da Sela, Ilhabela",no,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12979,42431,BR-0002,closed,Alegrete,-29.800278,-55.763054,<NA>,SA,BR,BR-RS,NaN,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12980,42432,BR-0003,closed,Fazenda Cana Brava,-17.418400,-39.581799,<NA>,SA,BR,BR-BA,NaN,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12981,42433,BR-0004,closed,Fazenda Sao Judas Tadeu,-12.617800,-60.900799,<NA>,SA,BR,BR-RO,NaN,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12982,42434,BR-0005,closed,Iguatemi,-23.632900,-54.629700,<NA>,SA,BR,BR-MS,NaN,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63946,42733,SWZT,small_airport,Fazenda do Brejo Airport,-19.254999,-44.473331,2451.0,SA,BR,BR-MG,Paraopeba,no,NaN,NaN,SWZT,MG0210,NaN,NaN,Ageo Agropecuária Ltda
63947,38197,SWZY,small_airport,Fazenda Sete Estrelas Airport,-11.576917,-58.234363,984.0,SA,BR,BR-MT,Brasnorte,no,NaN,NaN,SWZY,MT0451,NaN,NaN,NaN
63948,38198,SWZZ,heliport,Graer Heliport,-26.263237,-48.858604,39.0,SA,BR,BR-SC,Joinville,no,NaN,NaN,SWZZ,SC0101,NaN,NaN,NaN
81759,317829,XIG,small_airport,Xinguara Municipal Airport,-7.090600,-49.976500,810.0,SA,BR,BR-PA,Xinguara,no,NaN,XIG,SWSX,PA0150,NaN,https://en.wikipedia.org/wiki/Xinguara_Municip...,NaN


In [13]:
# restringindo ainda mais pra os aeroportos considerados grandes pelo dataset:
aeroportos_brasil_principais = aeroportos_brasil[aeroportos_brasil['type'] == 'large_airport']
display(aeroportos_brasil_principais)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
56493,5867,SBBE,large_airport,Val de Cans/Júlio Cezar Ribeiro International ...,-1.379279,-48.476207,54.0,SA,BR,BR-PA,Belém,yes,SBBE,BEL,SBBE,PA0001,http://www.infraero.gov.br/index.php/br/aeropo...,https://en.wikipedia.org/wiki/Val_de_Cans_Inte...,NaN
56499,5872,SBBR,large_airport,Presidente Juscelino Kubitschek International ...,-15.869167,-47.920834,3497.0,SA,BR,BR-DF,Brasília,yes,SBBR,BSB,SBBR,DF0001,http://www.infraero.gov.br/usa/aero_prev_home....,https://en.wikipedia.org/wiki/Bras%C3%ADlia_In...,NaN
56502,5875,SBBV,large_airport,Atlas Brasil Cantanhede International Airport,2.846150,-60.690648,276.0,SA,BR,BR-RR,Boa Vista,yes,SBBV,BVB,SBBV,RR0001,https://www.boavista-airport.com.br/,https://en.wikipedia.org/wiki/Boa_Vista_Intern...,Boa Vista International Airport
56509,5882,SBCF,large_airport,Tancredo Neves International Airport,-19.635710,-43.966928,2721.0,SA,BR,BR-MG,Belo Horizonte,yes,SBCF,CNF,SBCF,MG0001,http://www.infraero.gov.br/usa/aero_prev_home....,https://en.wikipedia.org/wiki/Belo_Horizonte_I...,http://www.infraero.gov.br/usa/aero_prev_home....
56519,5891,SBCT,large_airport,Curitiba-Afonso Pena International Airport,-25.528500,-49.175800,2988.0,SA,BR,BR-PR,Curitiba,yes,SBCT,CWB,SBCT,PR0001,https://www.ccraeroportos.com.br/curitiba-pr,https://en.wikipedia.org/wiki/Afonso_Pena_Inte...,NaN
56522,5894,SBCY,large_airport,Várzea Grande–Marechal Rondon International Ai...,-15.652900,-56.116699,617.0,SA,BR,BR-MT,Cuiabá,yes,SBCY,CGB,SBCY,MT0001,https://www.infraero.gov.br/index.php/aeroport...,https://en.wikipedia.org/wiki/Marechal_Rondon_...,NaN
56528,5897,SBEG,large_airport,Eduardo Gomes International Airport,-3.038610,-60.049702,264.0,SA,BR,BR-AM,Manaus,yes,SBEG,MAO,SBEG,AM0001,https://airport-manaus.com.br/,https://en.wikipedia.org/wiki/Eduardo_Gomes_In...,NaN
56533,5900,SBFI,large_airport,Cataratas International Airport,-25.594167,-54.489444,786.0,SA,BR,BR-PR,Foz do Iguaçu,yes,SBFI,IGU,SBFI,PR0002,https://aeroportos.motiva.com.br/foz-do-iguacu...,https://en.wikipedia.org/wiki/Foz_do_Igua%C3%A...,NaN
56534,5901,SBFL,large_airport,Hercílio Luz International Airport,-27.670279,-48.552502,16.0,SA,BR,BR-SC,Florianópolis,yes,SBFL,FLN,SBFL,SC0001,http://www.infraero.gov.br/usa/aero_prev_home....,https://en.wikipedia.org/wiki/Herc%C3%ADlio_Lu...,http://www.infraero.gov.br/usa/aero_prev_home....
56537,5905,SBFZ,large_airport,Pinto Martins International Airport,-3.775833,-38.532222,83.0,SA,BR,BR-CE,Fortaleza,yes,SBFZ,FOR,SBFZ,CE0001,https://fortaleza-airport.com.br/en,https://en.wikipedia.org/wiki/Pinto_Martins_In...,NaN


Nao temos representantes de todos os estados, então vou ter que adicionar aeroportos especificos pra alguns estados serem representados. Serão os principais de cada estado que faltou [Amapá(AP), Tocantins(TO), Mato Grosso do Sul(MS), Piauí(PI), e Sergipe(SE)]

In [24]:
# amapá:
aeroporto_amapa = aeroportos_brasil[aeroportos_brasil['iso_region'] == 'BR-AP']
aeroporto_amapa = aeroporto_amapa[aeroporto_amapa['type'] == 'medium_airport']
display(aeroporto_amapa)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
56583,5943,SBMQ,medium_airport,Macapá - Alberto Alcolumbre International Airport,0.050664,-51.072201,56.0,SA,BR,BR-AP,Macapá,yes,SBMQ,MCP,SBMQ,AP0001,https://www4.infraero.gov.br/aeroportos/aeropo...,https://en.wikipedia.org/wiki/Macap%C3%A1_Inte...,NaN
56597,5950,SBOI,medium_airport,Oiapoque Airport,3.854120,-51.797056,53.0,SA,BR,BR-AP,Oiapoque,no,SBOI,OYK,SBOI,AP0002,NaN,https://pt.wikipedia.org/wiki/Aeroporto_de_Oia...,NaN


In [ ]:
# o aeroporto alberto alcolumbre é maior, então ele será escolhido
aeroporto_amapa = aeroporto_amapa[aeroporto_amapa['id'] == 5943]
display(aeroporto_amapa)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
56583,5943,SBMQ,medium_airport,Macapá - Alberto Alcolumbre International Airport,0.050664,-51.072201,56.0,SA,BR,BR-AP,Macapá,yes,SBMQ,MCP,SBMQ,AP0001,https://www4.infraero.gov.br/aeroportos/aeropo...,https://en.wikipedia.org/wiki/Macap%C3%A1_Inte...,NaN


In [ ]:
# tocantins:
aeroporto_tocantins = aeroportos_brasil[aeroportos_brasil['iso_region'] == 'BR-TO']
aeroporto_tocantins = aeroporto_tocantins[aeroporto_tocantins['type'] == 'medium_airport']
display(aeroporto_tocantins)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
56617,5955,SBPJ,medium_airport,Brigadeiro Lysias Rodrigues Airport,-10.291500,-48.356998,774.0,SA,BR,BR-TO,Palmas,yes,SBPJ,PMW,SBPJ,TO0001,https://www4.infraero.gov.br/aeroportos/aeropo...,https://en.wikipedia.org/wiki/Palmas_Airport,NaN
56620,5958,SBPN,medium_airport,Porto Nacional Airport,-10.719402,-48.399700,870.0,SA,BR,BR-TO,Porto Nacional,no,NaN,PNB,SDPE,TO0003,NaN,https://en.wikipedia.org/wiki/Porto_Nacional_A...,SBPN
63619,538,SWGN,medium_airport,Araguaína Airport,-7.227870,-48.240501,771.0,SA,BR,BR-TO,Araguaína,yes,SWGN,AUX,SWGN,TO0002,NaN,https://en.wikipedia.org/wiki/Aragua%C3%ADna_A...,NaN


In [ ]:
# o aeroporto de palmas é maior, então ele será escolhido
aeroporto_tocantins = aeroporto_tocantins[aeroporto_tocantins['id'] == 5955]
display(aeroporto_tocantins)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
56617,5955,SBPJ,medium_airport,Brigadeiro Lysias Rodrigues Airport,-10.2915,-48.356998,774.0,SA,BR,BR-TO,Palmas,yes,SBPJ,PMW,SBPJ,TO0001,https://www4.infraero.gov.br/aeroportos/aeropo...,https://en.wikipedia.org/wiki/Palmas_Airport,NaN


In [29]:
# mato grosso do sul:
aeroporto_mgs = aeroportos_brasil[aeroportos_brasil['iso_region'] == 'BR-MS']
aeroporto_mgs = aeroporto_mgs[aeroporto_mgs['type'] == 'medium_airport']
display(aeroporto_mgs)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
56510,5883,SBCG,medium_airport,Campo Grande Airport,-20.469998,-54.673988,1833.0,SA,BR,BR-MS,Campo Grande,yes,SBCG,CGR,SBCG,MS0001,NaN,https://en.wikipedia.org/wiki/Campo_Grande_Int...,NaN
56518,5890,SBCR,medium_airport,Corumbá International Airport,-19.011930,-57.672772,463.0,SA,BR,BR-MS,Corumbá,yes,SBCR,CMG,SBCR,MS0009,NaN,https://en.wikipedia.org/wiki/Corumb%C3%A1_Int...,NaN
56622,5959,SBPP,medium_airport,Ponta Porã Airport,-22.549601,-55.702599,2156.0,SA,BR,BR-MS,Ponta Porã,yes,SBPP,PMG,SBPP,MS0005,https://www4.infraero.gov.br/aeroportos/aeropo...,https://en.wikipedia.org/wiki/Ponta_Por%C3%A3_...,NaN


In [30]:
# o aeroporto de campo grande é maior, então ele será escolhido
aeroporto_mgs = aeroporto_mgs[aeroporto_mgs['id'] == 5883]
display(aeroporto_mgs)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
56510,5883,SBCG,medium_airport,Campo Grande Airport,-20.469998,-54.673988,1833.0,SA,BR,BR-MS,Campo Grande,yes,SBCG,CGR,SBCG,MS0001,NaN,https://en.wikipedia.org/wiki/Campo_Grande_Int...,NaN


In [31]:
# piaui:
aeroporto_piaui = aeroportos_brasil[aeroportos_brasil['iso_region'] == 'BR-PI']
aeroporto_piaui = aeroporto_piaui[aeroporto_piaui['type'] == 'medium_airport']
display(aeroporto_piaui)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
56613,5952,SBPB,medium_airport,Parnaíba - Prefeito Doutor João Silva Filho In...,-2.89375,-41.731998,23.0,SA,BR,BR-PI,Parnaíba,yes,SBPB,PHB,SBPB,PI0002,NaN,https://en.wikipedia.org/wiki/Parna%C3%ADba-Pr...,Santos Dumont Airport
56672,5982,SBTE,medium_airport,Senador Petrônio Portela Airport,-5.06025,-42.823712,219.0,SA,BR,BR-PI,Teresina,yes,SBTE,THE,SBTE,PI0001,NaN,https://en.wikipedia.org/wiki/Teresina_Airport,NaN


In [34]:
# o aeroporto de teresina é maior, então ele será escolhido
aeroporto_piaui = aeroporto_piaui[aeroporto_piaui['id'] == 5982]
display(aeroporto_piaui)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
56672,5982,SBTE,medium_airport,Senador Petrônio Portela Airport,-5.06025,-42.823712,219.0,SA,BR,BR-PI,Teresina,yes,SBTE,THE,SBTE,PI0001,NaN,https://en.wikipedia.org/wiki/Teresina_Airport,NaN


In [33]:
# sergipe:
aeroporto_sergipe = aeroportos_brasil[aeroportos_brasil['iso_region'] == 'BR-SE']
aeroporto_sergipe = aeroporto_sergipe[aeroporto_sergipe['type'] == 'medium_airport']
display(aeroporto_sergipe)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
56489,5862,SBAR,medium_airport,Aracaju - Santa Maria Airport,-10.98394,-37.072873,23.0,SA,BR,BR-SE,Aracaju,yes,SBAR,AJU,SBAR,SE0001,https://www.aenabrasil.com.br/pt/aeroportos/ae...,https://en.wikipedia.org/wiki/Santa_Maria_Airp...,NaN


In [35]:
# o aeroporto de aracaju é maior, então ele será escolhido
aeroporto_sergipe = aeroporto_sergipe[aeroporto_sergipe['id'] == 5862]
display(aeroporto_sergipe)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
56489,5862,SBAR,medium_airport,Aracaju - Santa Maria Airport,-10.98394,-37.072873,23.0,SA,BR,BR-SE,Aracaju,yes,SBAR,AJU,SBAR,SE0001,https://www.aenabrasil.com.br/pt/aeroportos/ae...,https://en.wikipedia.org/wiki/Santa_Maria_Airp...,NaN


Agora, todos os estados que faltavam tiveram seus aeroportos escolhidos então resta adiciona-los ao dataset principal


In [36]:
# adicionando os escolhidos:
dataset_final = pd.concat([aeroportos_brasil_principais, aeroporto_amapa, aeroporto_mgs, aeroporto_piaui, aeroporto_sergipe, aeroporto_tocantins])
display(dataset_final)

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
56493,5867,SBBE,large_airport,Val de Cans/Júlio Cezar Ribeiro International ...,-1.379279,-48.476207,54.0,SA,BR,BR-PA,Belém,yes,SBBE,BEL,SBBE,PA0001,http://www.infraero.gov.br/index.php/br/aeropo...,https://en.wikipedia.org/wiki/Val_de_Cans_Inte...,NaN
56499,5872,SBBR,large_airport,Presidente Juscelino Kubitschek International ...,-15.869167,-47.920834,3497.0,SA,BR,BR-DF,Brasília,yes,SBBR,BSB,SBBR,DF0001,http://www.infraero.gov.br/usa/aero_prev_home....,https://en.wikipedia.org/wiki/Bras%C3%ADlia_In...,NaN
56502,5875,SBBV,large_airport,Atlas Brasil Cantanhede International Airport,2.846150,-60.690648,276.0,SA,BR,BR-RR,Boa Vista,yes,SBBV,BVB,SBBV,RR0001,https://www.boavista-airport.com.br/,https://en.wikipedia.org/wiki/Boa_Vista_Intern...,Boa Vista International Airport
56509,5882,SBCF,large_airport,Tancredo Neves International Airport,-19.635710,-43.966928,2721.0,SA,BR,BR-MG,Belo Horizonte,yes,SBCF,CNF,SBCF,MG0001,http://www.infraero.gov.br/usa/aero_prev_home....,https://en.wikipedia.org/wiki/Belo_Horizonte_I...,http://www.infraero.gov.br/usa/aero_prev_home....
56519,5891,SBCT,large_airport,Curitiba-Afonso Pena International Airport,-25.528500,-49.175800,2988.0,SA,BR,BR-PR,Curitiba,yes,SBCT,CWB,SBCT,PR0001,https://www.ccraeroportos.com.br/curitiba-pr,https://en.wikipedia.org/wiki/Afonso_Pena_Inte...,NaN
56522,5894,SBCY,large_airport,Várzea Grande–Marechal Rondon International Ai...,-15.652900,-56.116699,617.0,SA,BR,BR-MT,Cuiabá,yes,SBCY,CGB,SBCY,MT0001,https://www.infraero.gov.br/index.php/aeroport...,https://en.wikipedia.org/wiki/Marechal_Rondon_...,NaN
56528,5897,SBEG,large_airport,Eduardo Gomes International Airport,-3.038610,-60.049702,264.0,SA,BR,BR-AM,Manaus,yes,SBEG,MAO,SBEG,AM0001,https://airport-manaus.com.br/,https://en.wikipedia.org/wiki/Eduardo_Gomes_In...,NaN
56533,5900,SBFI,large_airport,Cataratas International Airport,-25.594167,-54.489444,786.0,SA,BR,BR-PR,Foz do Iguaçu,yes,SBFI,IGU,SBFI,PR0002,https://aeroportos.motiva.com.br/foz-do-iguacu...,https://en.wikipedia.org/wiki/Foz_do_Igua%C3%A...,NaN
56534,5901,SBFL,large_airport,Hercílio Luz International Airport,-27.670279,-48.552502,16.0,SA,BR,BR-SC,Florianópolis,yes,SBFL,FLN,SBFL,SC0001,http://www.infraero.gov.br/usa/aero_prev_home....,https://en.wikipedia.org/wiki/Herc%C3%ADlio_Lu...,http://www.infraero.gov.br/usa/aero_prev_home....
56537,5905,SBFZ,large_airport,Pinto Martins International Airport,-3.775833,-38.532222,83.0,SA,BR,BR-CE,Fortaleza,yes,SBFZ,FOR,SBFZ,CE0001,https://fortaleza-airport.com.br/en,https://en.wikipedia.org/wiki/Pinto_Martins_In...,NaN


In [43]:
# verificaçõão final de duplicatas em name (que eu adiei mais cedo):
mostrar_duplicatas(dataset_final, chave_busca='name')

print("-------------------------------------")

# verificaçõão final de tipos: (o 'elevation_ft' está com tipo object oque nao parece certo pra mim, pois deveria ser um numero)
print(dataset_final.dtypes)   #(mas como essa coluna não será usada no projeto, não irei corrigir ela)

✅ Nenhuma duplicata encontrada! | Tipo de busca: Nas colunas: name
-------------------------------------
id                     int64
ident                    str
type                     str
name                     str
latitude_deg         float64
longitude_deg        float64
elevation_ft          object
continent                str
iso_country              str
iso_region               str
municipality             str
scheduled_service        str
icao_code                str
iata_code                str
gps_code                 str
local_code               str
home_link                str
wikipedia_link           str
keywords                 str
dtype: object


In [ ]:
# por fim, limpando o dataset final pra incluir apenas as colunas que eu julguei importantes
# mesmo que nao fossem ser utilizadas diretamente em calculos ou classificações no algoritmo
# id = identificador numérico
# ident = identificador textual
# name = nome
# municipality = cidade
# iso_region = estado
# latitude_deg = latitude
# longitude_deg = longitude
colunas_importantes = ['id', 'ident','name', 'municipality', 'iso_region', 'latitude_deg', 'longitude_deg']
dataset_final_clean = dataset_final[colunas_importantes]

display(dataset_final_clean)

,id,ident,name,municipality,iso_region,latitude_deg,longitude_deg
56493,5867,SBBE,Val de Cans/Júlio Cezar Ribeiro International ...,Belém,BR-PA,-1.379279,-48.476207
56499,5872,SBBR,Presidente Juscelino Kubitschek International ...,Brasília,BR-DF,-15.869167,-47.920834
56502,5875,SBBV,Atlas Brasil Cantanhede International Airport,Boa Vista,BR-RR,2.846150,-60.690648
56509,5882,SBCF,Tancredo Neves International Airport,Belo Horizonte,BR-MG,-19.635710,-43.966928
56519,5891,SBCT,Curitiba-Afonso Pena International Airport,Curitiba,BR-PR,-25.528500,-49.175800
56522,5894,SBCY,Várzea Grande–Marechal Rondon International Ai...,Cuiabá,BR-MT,-15.652900,-56.116699
56528,5897,SBEG,Eduardo Gomes International Airport,Manaus,BR-AM,-3.038610,-60.049702
56533,5900,SBFI,Cataratas International Airport,Foz do Iguaçu,BR-PR,-25.594167,-54.489444
56534,5901,SBFL,Hercílio Luz International Airport,Florianópolis,BR-SC,-27.670279,-48.552502
56537,5905,SBFZ,Pinto Martins International Airport,Fortaleza,BR-CE,-3.775833,-38.532222


In [45]:
# salvando o dataset tratado na pasta:
dataset_final_clean.to_csv("dataset_final_clean.csv", index=False)